# Edge-AI Battery Monitoring System
## SOC, SOH, RUL and ESP32-S3 Data Augmentation

**Project:** Edge-AI based smart battery monitoring system  
**Target:** ESP32-S3 with calibrated voltage taps, NTC thermistors and INA226 current sensing  
**Dataset:** NASA Li-ion Battery Degradation Dataset  
**Purpose:** A concise, reproducible summary for project review

---

### Study questions

1. Can a small deep neural network estimate **State of Charge (SOC)** from voltage, current and temperature?
2. Can a small deep neural network estimate **State of Health (SOH)** from cycle-level degradation features?
3. How will **Remaining Useful Life (RUL)** be implemented?
4. How will measurements from `BATTERY_MONITORING_S3` reduce the domain gap between NASA laboratory data and the real battery-monitoring hardware?


## 1. System boundary and terminology

The implementation is divided into three connected layers:

| Layer | Inputs | Output | Current status |
|---|---|---|---|
| Hardware sensing | Three voltage taps, three NTC channels, INA226 current | Timestamped CSV telemetry | Implemented in ESP32-S3 firmware |
| Data/model analysis | NASA discharge cycles and later hardware CSV data | SOC, SOH and RUL models | SOC and SOH experiments available |
| Dashboard/deployment | Model estimates and health indicators | User-facing battery status | Presentation/demo layer; live service is not yet established |

**Important distinction:** the firmware currently calculates a simple voltage-based SOC estimate for monitoring and logging. It does **not** run the NASA-trained DNN on the ESP32-S3 yet.


## 2. Data sources and labels

### NASA data

- `timeseries_soc_FINAL.csv` / equivalent cleaned SOC time series: timestep-level voltage, current, temperature and SOC.
- `cycle_summary_soh_rul_FINAL.csv`: one row per discharge cycle with capacity, electrical/thermal summary features, SOH and RUL.
- `cycle_summary_soh_rul_WITH_IMPEDANCE.csv`: the same cycle summary with NASA impedance features where available.

### Labels

- **SOC:** coulomb-counting label within each discharge cycle, truncated at the first SOC-zero crossing.
- **SOH:** capacity relative to a robust battery-specific rated-capacity baseline.
- **RUL:** number of discharge cycles remaining until the selected end-of-life threshold. In this study the operational threshold is **SOH <= 0.70** unless a different threshold is explicitly reported.

### Split rule

All reported evaluation must split by `battery_id`, not by randomly splitting individual rows. Holding out complete batteries tests generalisation to an unseen battery and avoids leakage from highly correlated cycles.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the repository data directory whether the notebook is opened from
# the notebook folder or from the repository root.
_candidates = [Path.cwd(), *Path.cwd().parents]
DATA_ROOT = next((p / 'NASA Dataset' for p in _candidates if (p / 'NASA Dataset').is_dir()), None)
if DATA_ROOT is None:
    DATA_ROOT = Path('..').resolve().parent

SOC_PATH = DATA_ROOT / 'Model' / 'SoC_study' / 'timeseries_soc_FINAL.csv'
SOH_PATH = DATA_ROOT / 'cycle_summary_soh_rul_FINAL.csv'
SOH_IMP_PATH = DATA_ROOT / 'cycle_summary_soh_rul_WITH_IMPEDANCE.csv'

print('Data root:', DATA_ROOT.resolve())
print('SOC file exists:', SOC_PATH.exists())
print('SOH file exists:', SOH_PATH.exists())
print('SOH + impedance file exists:', SOH_IMP_PATH.exists())


## 3. SOC DNN experiment

The SOC model uses a short temporal window of the three signals most likely to be available on the hardware:

- pack/cell voltage
- signed current
- temperature

The window is resampled to a regular time grid. A compact DNN/MLP receives the flattened window, while an LSTM receives the sequence directly. Standardisation is fitted on the training batteries only.

### Validated SOC results

The following values are from the cleaned 32-battery SOC experiment in `02_SOC_estimation_FINAL_PRESENTATION.ipynb`. The evaluation used 32 batteries, a regularised/resampled time series, 12-step windows with voltage/current/temperature, and a battery-level held-out test set containing B0029, B0030, B0038, B0040, B0047 and B0054. SOC metrics are in percentage points.

| Model | MAE (%) | RMSE (%) | R2 | Role |
|---|---:|---:|---:|---|
| Coulomb counting | 4.266 | 7.540 | 0.9273 | Classical baseline |
| Random Forest | 3.587 | 5.152 | 0.9661 | Strongest SOC comparator |
| XGBoost | 3.964 | 5.401 | 0.9627 | Additional tree-based comparator |
| Simplified MLP / DNN | 7.591 | 9.693 | 0.8799 | Initial compact neural model |
| Fine-tuned MLP / DNN | 4.647 | 6.274 | 0.9497 | Preferred neural SOC candidate |
| LSTM | 7.070 | 9.275 | 0.8900 | Sequence-model comparator |

The fine-tuned DNN substantially improves on the initial simplified MLP: MAE decreases from 7.591% to 4.647%, RMSE decreases from 9.693% to 6.274%, and R2 increases from 0.8799 to 0.9497. Random Forest remains the best numerical SOC model in this experiment, but the intended deployment direction is the fine-tuned DNN because it has a compact dense architecture and a direct path toward quantised TFLite Micro inference. XGBoost and Random Forest remain useful offline references.

### SOC selection decision

The project will proceed with the **fine-tuned MLP/DNN** as the primary SOC deployment candidate. This is an engineering decision rather than a claim that it has the lowest current error. The model must still be re-evaluated after hardware-informed augmentation and quantisation, using the same held-out battery protocol.


In [ ]:
# Minimal SOC data inspection. This cell is intentionally lightweight; it does not retrain a neural network.
if SOC_PATH.exists():
    soc = pd.read_csv(SOC_PATH)
    print('SOC rows:', len(soc))
    print('SOC columns:', list(soc.columns))
    if 'battery_id' in soc and 'soc' in soc:
        print('Batteries:', soc['battery_id'].nunique())
        print(soc[['soc']].describe().round(3))
else:
    print('Place the cleaned SOC CSV at:', SOC_PATH)


### SOC interpretation

The DNN results are useful for a first deployment study, but they do not eliminate the domain gap:

- NASA discharge currents are concentrated at a few constant-current levels.
- Real ESP32 operation produces variable and pulsed load current.
- Battery-to-battery ageing shifts the voltage-SOC relationship.
- The hardware's voltage calibration, sensor noise and wiring introduce additional variation.

The ESP32-S3 measurements are therefore used to make the training distribution more representative of deployment conditions before attempting quantised inference.


## 4. SOH DNN experiment

SOH is estimated once per discharge cycle rather than at every sensor timestep. The compact DNN uses cycle-level features that can be calculated from a logged discharge segment:

- cycle position as a percentage of the battery life observed so far
- voltage statistics
- current statistics
- temperature statistics
- discharge duration
- rated-capacity and capacity-ratio features

The train/test split is battery-level using `GroupShuffleSplit`.

### Validated SOH results

The following values are from `SoH_Fine.ipynb`, using 32 valid batteries and 2,524 cycle rows. The same battery-level split was used for both models: 25 training batteries and 7 held-out test batteries (B0029, B0030, B0038, B0040, B0047, B0054 and B0055). The reported SOH values are fractions, where `1.0` represents approximately 100% health.

| Model | MAE | RMSE | R2 | Approximate MAE in percentage points | Role |
|---|---:|---:|---:|---:|---|
| XGBoost | 0.0072 | 0.0270 | 0.9494 | 0.72 | Best offline SOH comparator |
| Simplified SOH DNN | 0.0194 | 0.0320 | 0.9287 | 1.94 | Intended embedded model family |

XGBoost achieves lower SOH error in the current offline experiment. Nevertheless, the project intends to proceed with the **SOH DNN** so that SOC, SOH and the future RUL estimator can share a neural-network deployment and quantisation workflow on the ESP32-S3. XGBoost remains the benchmark to beat and may be retained for server-side or offline analysis.

### SOH feature set and model structure

The DNN uses 13 cycle-level features: cycle percentage, maximum/minimum/mean voltage, maximum/minimum/mean current, maximum/minimum/mean temperature, discharge duration, rated capacity and capacity ratio. The compact network is `Dense(24, relu) -> Dropout(0.1) -> Dense(12, relu) -> Dropout(0.1) -> Dense(1, linear)`, trained with Adam, MSE loss, batch size 32, early stopping and training-only standardisation.


In [1]:
# Reproduce the SOH feature construction and verify the schema.
if SOH_PATH.exists():
    soh = pd.read_csv(SOH_PATH)
    soh['cycle_max'] = soh.groupby('battery_id')['cycle'].transform('max')
    soh['cycle_pct'] = soh['cycle'] / soh['cycle_max']
    soh['capacity_ratio'] = soh['capacity_Ah'] / soh['rated_capacity_Ah']
    soh_features = [
        'cycle_pct', 'max_voltage_V', 'min_voltage_V', 'mean_voltage_V',
        'max_current_A', 'min_current_A', 'mean_current_A',
        'max_temperature_C', 'min_temperature_C', 'mean_temperature_C',
        'discharge_time_s', 'rated_capacity_Ah', 'capacity_ratio'
    ]
    missing = [c for c in soh_features + ['soh', 'battery_id'] if c not in soh.columns]
    print('SOH rows:', len(soh))
    print('Batteries:', soh['battery_id'].nunique())
    print('Missing required columns:', missing)
else:
    print('Place the SOH summary CSV at:', SOH_PATH)


NameError: name 'SOH_PATH' is not defined

## 5. RUL implementation plan

RUL is not a separate arbitrary regression target. It is derived from the degradation trajectory and must be defined against an explicit end-of-life rule.

### Training target

For each battery:

1. Sort cycles by `cycle`.
2. Identify the first cycle where `soh <= 0.70`.
3. Define `eol_cycle` as that cycle.
4. For every earlier cycle, calculate:

```text
RUL(cycle) = max(eol_cycle - cycle, 0)
```

If a battery does not reach the threshold in the available experiment, it is **right-censored**. Those batteries must not be silently treated as if their final observed cycle were the true EOL.

### Model inputs

The first RUL model can reuse the SOH feature vector plus temporal features:

- current cycle number and cycle percentage
- latest SOH estimate
- recent SOH slope over a fixed cycle window
- capacity fade rate
- discharge duration trend
- mean/min voltage and temperature
- `Re_ohm` and `Rct_ohm` when impedance is available

### Deployment flow

```text
new discharge segment
        -> feature extraction
        -> SOH DNN estimate
        -> update recent degradation history
        -> RUL DNN or sequence model
        -> cycles remaining + confidence/range
```

A practical first version is a small MLP for RUL using a fixed history window. A later version can use an LSTM/temporal model if the number of labelled batteries is increased. RUL should be reported with uncertainty and a clear statement when the input is outside the NASA training distribution.


In [ ]:
# Reference implementation for constructing a non-censored RUL target.
# This is a definition/check cell, not a trained RUL model.
def add_rul_target(frame, eol_soh=0.70):
    frame = frame.sort_values(['battery_id', 'cycle']).copy()
    eol = (frame[frame['soh'] <= eol_soh]
           .groupby('battery_id', as_index=False)['cycle']
           .min()
           .rename(columns={'cycle': 'eol_cycle'}))
    frame = frame.merge(eol, on='battery_id', how='left')
    frame['rul_observed'] = (frame['eol_cycle'] - frame['cycle']).clip(lower=0)
    frame['rul_is_censored'] = frame['eol_cycle'].isna()
    return frame

if SOH_PATH.exists():
    rul_check = add_rul_target(pd.read_csv(SOH_PATH))
    print(rul_check[['battery_id', 'cycle', 'soh', 'eol_cycle', 'rul_observed', 'rul_is_censored']].head())
    print('Censored batteries:', rul_check.loc[rul_check['rul_is_censored'], 'battery_id'].nunique())


## 6. ESP32-S3 logging contract

The current firmware emits a CSV stream with the following fields:

```text
battery_id, cycle, t_s, voltage, current, temperature, soc,
cell1_v, cell2_v, cell3_v, temp1_c, temp2_c, temp3_c, cycle_capacity_Ah
```

The values are produced from:

- calibrated quadratic ADC equations for the three cumulative voltage taps
- subtraction of cumulative taps to obtain individual cell voltages
- averaged NTC readings using the configured beta equation
- INA226 current and power measurements
- integrated current for `cycle_capacity_Ah`

The firmware's `soc` column is a **monitoring estimate**, not an independent ground-truth label. It must not be used as the supervised SOC target when augmenting NASA data.


In [ ]:
FIRMWARE_COLUMNS = [
    'battery_id', 'cycle', 't_s', 'voltage', 'current', 'temperature', 'soc',
    'cell1_v', 'cell2_v', 'cell3_v', 'temp1_c', 'temp2_c', 'temp3_c',
    'cycle_capacity_Ah'
]

def validate_firmware_csv(frame):
    missing = [c for c in FIRMWARE_COLUMNS if c not in frame.columns]
    if missing:
        raise ValueError(f'Missing firmware columns: {missing}')
    out = frame.copy()
    out['battery_id'] = out['battery_id'].astype(str)
    out['cycle'] = pd.to_numeric(out['cycle'], errors='coerce')
    for col in FIRMWARE_COLUMNS[2:]:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    return out.sort_values(['battery_id', 'cycle', 't_s'])

# Example usage after copying a serial capture into this folder:
# hardware = validate_firmware_csv(pd.read_csv('hardware_capture.csv'))
# display(hardware.head())


## 7. Hardware-informed augmentation strategy

The goal is not to invent labels. The goal is to expose the DNN to the sensor and load conditions that the ESP32-S3 actually produces.

### Step A ? collect and qualify hardware data

1. Capture the firmware CSV during controlled discharge experiments.
2. Record battery identity, cycle number, sampling interval and test conditions.
3. Reject rows with impossible voltages, broken timestamps or missing sensor values.
4. Keep raw captures unchanged; write a separate cleaned file.

### Step B ? estimate deployment-domain variation

From qualified hardware captures, estimate distributions for:

- voltage measurement residual/noise
- current noise and pulsed-load transitions
- temperature noise and drift
- sampling interval and short gaps
- cell imbalance patterns

### Step C ? augment labelled NASA windows

For each NASA SOC window, create hardware-informed variants by applying bounded perturbations sampled from those measured distributions:

```text
voltage_aug     = voltage_nasa     + measured_voltage_noise
current_aug     = current_nasa     + measured_current_noise
temperature_aug = temperature_nasa + measured_temperature_noise
```

The original NASA SOC label is retained only when the perturbation represents measurement/domain variation rather than a physically different battery state. Current sign and units must remain consistent with the firmware convention.

### Step D ? do not misuse hardware SOC

The firmware's voltage-based SOC is useful for monitoring and quality checks, but it is not a trusted label for supervised learning because it is itself an estimate. For a stronger future dataset, pair each hardware capture with a reference SOC obtained by calibrated coulomb counting and known initial capacity.

### SOH/RUL augmentation limitation

Hardware telemetry alone does not provide SOH or RUL labels. To augment SOH/RUL training, hardware experiments must include repeated cycles and a capacity/reference test. Otherwise, hardware data can be used for input-domain adaptation and validation, but not as labelled degradation history.


In [ ]:
# Reusable, bounded augmentation template.
# The noise values below are placeholders until measured residual distributions are supplied.
def augment_soc_inputs(nasa_windows, noise_std, seed=42, copies=3):
    """Create input-only variants; labels remain the original NASA labels."""
    rng = np.random.default_rng(seed)
    variants = [nasa_windows]
    for _ in range(copies):
        noise = rng.normal(0.0, np.asarray(noise_std), size=nasa_windows.shape)
        variants.append(nasa_windows + noise)
    return np.concatenate(variants, axis=0)

# Example shape: (samples, window_length, 3) for [voltage, current, temperature].
# Use standard deviations measured from qualified ESP32-S3 captures, not arbitrary values:
# augmented_X = augment_soc_inputs(X_train, noise_std=[sigma_v, sigma_i, sigma_t])
# augmented_y = np.tile(y_train, 1 + copies)


## 8. Deployment roadmap

### Phase 1 ? complete the evidence

- Reconcile the SOC metrics with one named preprocessing version and one held-out battery list.
- Re-run the SOH DNN from a clean notebook and save the exact split and metrics.
- Collect qualified ESP32-S3 variable-load captures.
- Re-calibrate the Tap1 and Tap3 channels before safety-critical use; Tap3 currently has the largest documented error.

### Phase 2 ? model development

- Estimate hardware noise/load distributions.
- Train SOC with hardware-informed input augmentation.
- Train SOH on cycle-level capacity and sensor features.
- Construct RUL targets with explicit EOL and censoring treatment.
- Evaluate all models by held-out battery, not random row.

### Phase 3 ? embedded deployment

- Select the smallest model meeting the accuracy requirement.
- Quantise the DNN to int8 using a representative dataset containing both NASA and hardware-domain samples.
- Validate quantised inference against the floating-point model.
- Integrate inference only after the data logger and calibration checks are stable.

### Final message

The current project has a working sensing and logging foundation, validated SOC/SOH DNN experiments, and a clear RUL formulation. The next scientific milestone is not simply adding more synthetic rows; it is collecting real ESP32-S3 variable-load data and using it to make the model's input distribution representative of deployment.
